# Edge-consistent RFQ responder

This notebook quotes synthetic RFQs two ways:

1. **Edge-consistent**: solve `quoted_edge = selection + costs + target_edge` so a fill earns exactly the target net edge.
2. **Expected-PnL**: scan quoted edge and keep the value that maximizes `P(fill) * dollar net edge`.

The consistent quote does not trade fill probability against markup. The expected-PnL quote does.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rfq_edge import (
    default_config,
    demo_book_spec,
    evaluate_quote,
    generate_rfq_book,
    maximize_expected_pnl,
    run_book,
    solve_consistent_edge,
)

## Synthetic RFQ history

Draw a reproducible book of fake RFQ rows. Each row carries CP+ and internal mids, the dealer quote, side, win flag, t+5 clean mark, and bond/issuer metadata so the demo can run before live data is connected.

In [ ]:
from rfq_edge import to_rfq_request

spec = demo_book_spec()
book = generate_rfq_book(spec)
record = book[0]
request = to_rfq_request(record, spec, 0)
config = default_config()

print(f"book_size={len(book)}")
print(
    f"{record.rfq_id} | {record.bond.issuer.issuer_name} {record.bond.bond_id} "
    f"| side={record.side.value} | cp+={record.cp_plus_mid:.4f} "
    f"| internal={record.internal_mid:.4f} | quote={record.quote:.4f} "
    f"| won={record.quote_won} | t+5={record.t5_clean_mark:.4f}"
)

## Fill, selection, and expected PnL versus quoted edge

Wider dealer edge lowers fill odds and also lowers adverse selection. Expected PnL is the product of fill odds and dollar net edge.

In [ ]:
def edge_grid_bps():
    start = config.search.min_edge_bps
    stop = config.search.max_edge_bps
    step = config.search.step_bps
    values = []
    current = start
    while current <= stop + 1e-12:
        values.append(round(current, 10))
        current += step
    return values


grid = edge_grid_bps()
curves = [evaluate_quote(request, config.models, edge) for edge in grid]
consistent = solve_consistent_edge(request, config.models, config.search)
optimal = maximize_expected_pnl(request, config.models, config.search)

print("edge_consistent", consistent.components)
print("expected_pnl", optimal.components)

In [ ]:
import matplotlib.pyplot as plt

fill = [row.fill_probability for row in curves]
selection = [row.selection_bps for row in curves]
required = [row.required_edge_bps for row in curves]
pnl = [row.expected_pnl for row in curves]
quoted = [row.quoted_edge_bps for row in curves]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

axes[0].plot(quoted, fill, color="black")
axes[0].set_title("Fill probability")
axes[0].set_xlabel("quoted edge (bps)")
axes[0].set_ylabel("P(fill)")

axes[1].plot(quoted, selection, color="black", label="selection")
axes[1].plot(quoted, required, color="gray", linestyle="--", label="required edge")
axes[1].plot(quoted, quoted, color="0.6", linestyle=":", label="quoted edge")
axes[1].set_title("Selection and consistency")
axes[1].set_xlabel("quoted edge (bps)")
axes[1].set_ylabel("bps")
axes[1].legend(frameon=False)

axes[2].plot(quoted, pnl, color="black")
axes[2].axvline(
    consistent.components.quoted_edge_bps,
    color="gray",
    linestyle="--",
    label="consistent",
)
axes[2].axvline(
    optimal.components.quoted_edge_bps,
    color="black",
    linestyle=":",
    label="expected pnl",
)
axes[2].set_title("Expected PnL")
axes[2].set_xlabel("quoted edge (bps)")
axes[2].set_ylabel("dollars")
axes[2].legend(frameon=False)

fig.tight_layout()
plt.show()

## Book-level decisions

Run both rules on the full synthetic book. The consistent quote's net edge should match the configured target on every name.

In [ ]:
decisions = run_book(
    tuple(to_rfq_request(record, spec, index) for index, record in enumerate(book)),
    config,
)
target = config.models.target_edge_bps

print(
    f"{'rfq_id':<12} {'issuer':<18} {'side':<5} {'cons_bps':>8} {'opt_bps':>8} "
    f"{'cons_net':>8} {'opt_net':>8} {'cons_pnl':>10} {'opt_pnl':>10}"
)
for decision in decisions[:12]:
    cons = decision.consistent.components
    opt = decision.optimal.components
    issuer = next(
        record.bond.issuer.issuer_name
        for record in book
        if record.rfq_id == decision.request.rfq_id
    )
    print(
        f"{decision.request.rfq_id:<12} {issuer:<18} {decision.request.side.value:<5} "
        f"{cons.quoted_edge_bps:8.2f} {opt.quoted_edge_bps:8.2f} "
        f"{cons.net_edge_bps:8.2f} {opt.net_edge_bps:8.2f} "
        f"{cons.expected_pnl:10.2f} {opt.expected_pnl:10.2f}"
    )

max_net_gap = max(
    abs(decision.consistent.components.net_edge_bps - target)
    for decision in decisions
)
print(f"max |consistent net edge - target| = {max_net_gap:.6f} bps")